# Forecast model selection + diagnostics

Production currently runs `GradientBoostingRegressor` on lag + calendar features. Before locking that in I want to:

1. Compare it against simpler baselines and one obvious alternative (LightGBM).
2. Try a statistical model (SARIMA) since the series is daily with weekly seasonality.
3. Inspect residuals on the chosen model — if they're not white-noise-ish, the model is leaving structure on the table.
4. Simulate drift: train on 2010 only, score 2011, and see how badly MAE degrades. This tells me if periodic retraining is needed.

If the gradient boosting model wins by a meaningful margin **and** the residuals look clean, I keep it. Otherwise I revisit.

In [ ]:
from pathlib import Path
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore", category=FutureWarning)
plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

## Load + reuse the production data pipeline

I import the same helpers the service uses so the notebook can't drift from production behaviour.

In [ ]:
import sys
sys.path.insert(0, str(Path("..") / "backend"))

from app.services.data_service import load_transactions
from app.services.forecast_service import (
    add_features,
    build_daily_revenue,
    FEATURE_NAMES,
    MODEL_PARAMS,
)

df = load_transactions()
series = build_daily_revenue(df)
print(f"Series length: {len(series):,} days")
print(f"Range: {series.index.min().date()} → {series.index.max().date()}")

## Evaluation harness

Chronological 80/20 split. All models score on the same held-out window so the comparison is apples-to-apples.

In [ ]:
split_at = int(len(series) * 0.8)
train_series = series.iloc[:split_at]
test_series = series.iloc[split_at:]
print(f"Train: {len(train_series)} days  ({train_series.index.min().date()} → {train_series.index.max().date()})")
print(f"Test : {len(test_series)} days  ({test_series.index.min().date()} → {test_series.index.max().date()})")

def metrics(actual: pd.Series, predicted: pd.Series) -> dict:
    actual = np.asarray(actual)
    predicted = np.asarray(predicted)
    mae = float(np.mean(np.abs(actual - predicted)))
    rmse = float(np.sqrt(np.mean((actual - predicted) ** 2)))
    mask = actual != 0
    mape = float(np.mean(np.abs((actual[mask] - predicted[mask]) / actual[mask])) * 100) if mask.any() else 0.0
    return {"MAE": mae, "RMSE": rmse, "MAPE": mape}

## Baselines

Before training anything fancy I want a floor. If a tree-based model can't beat "predict last week's value", something is wrong.

Three baselines:

- **Mean of training:** constant value.
- **Naive (lag 1):** predict yesterday's value for every day.
- **Seasonal naive (lag 7):** predict last week's same weekday — should catch the weekly seasonality.

In [ ]:
results = {}

# Mean baseline
mean_val = train_series.mean()
results["Mean baseline"] = metrics(test_series, [mean_val] * len(test_series))

# Naive (yesterday). We need the actual value the day before each test point.
lag1 = series.shift(1).loc[test_series.index]
results["Naive (lag-1)"] = metrics(test_series, lag1)

# Seasonal naive (lag 7).
lag7 = series.shift(7).loc[test_series.index]
results["Seasonal naive (lag-7)"] = metrics(test_series, lag7)

pd.DataFrame(results).T.round(2)

Seasonal naive should beat lag-1 if the weekly seasonality is real. If it doesn't, my mental model of the data is off.

## SARIMA

Daily series, weekly seasonality → SARIMA(p,d,q)(P,D,Q,7) is a natural fit. I keep the order modest because this dataset has only ~24 months — fancy parameters tend to overfit.

In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

sarima = SARIMAX(
    train_series,
    order=(1, 1, 1),
    seasonal_order=(1, 1, 1, 7),
    enforce_stationarity=False,
    enforce_invertibility=False,
).fit(disp=False)

sarima_pred = sarima.forecast(steps=len(test_series))
sarima_pred.index = test_series.index
results["SARIMA(1,1,1)(1,1,1,7)"] = metrics(test_series, sarima_pred)
pd.DataFrame(results).T.round(2)

## Tree-based models

Same features as production: day-of-week, month, week-of-year, weekend flag, lag 1 + 7, rolling 7/14/30.

In [ ]:
features = add_features(series)
train_features = features.loc[features.index < test_series.index[0]]
test_features = features.loc[features.index >= test_series.index[0]]

x_train = train_features[FEATURE_NAMES]
y_train = train_features["revenue"]
x_test = test_features[FEATURE_NAMES]
y_test = test_features["revenue"]

print(f"Train rows: {len(x_train)}  Test rows: {len(x_test)}")

In [ ]:
from sklearn.ensemble import GradientBoostingRegressor, RandomForestRegressor
import lightgbm as lgb

models = {
    "GradientBoosting (prod)": GradientBoostingRegressor(**MODEL_PARAMS),
    "RandomForest": RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42, n_jobs=1),
    "LightGBM": lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42, verbose=-1),
}

predictions = {}
for name, model in models.items():
    model.fit(x_train, y_train)
    pred = model.predict(x_test)
    predictions[name] = pred
    results[name] = metrics(y_test, pred)

pd.DataFrame(results).T.round(2).sort_values("MAE")

**Reading the table:**

- If gradient boosting wins by < 5% MAE over seasonal naive, the model is barely earning its keep. The right call would be to keep things simple.
- If LightGBM wins by > 10% it's worth switching, but only if I can also keep the runtime story clean (single training script, joblib artifact).
- SARIMA tends to perform well on short, low-noise series; if it wins here that's a story worth telling on the methodology page.

In [ ]:
# Visual comparison on the last 60 test days — easier to eyeball than the
# full window.
fig, ax = plt.subplots(figsize=(12, 4))
tail = y_test.iloc[-60:]
ax.plot(tail.index, tail.values, label="Actual", color="#0f172a", linewidth=2)
for name, pred in predictions.items():
    ax.plot(tail.index, pred[-60:], label=name, alpha=0.7, linewidth=1.2)
ax.set_title("Last 60 days — actual vs predicted")
ax.set_ylabel("Revenue")
ax.legend(loc="upper left", fontsize=9)
plt.tight_layout(); plt.show()

## Residual diagnostics on the production model

If residuals show structure (trend, seasonality, fat tails), there's information the model isn't capturing.

I check four things:

1. **Residual time series** — should look random around zero.
2. **ACF / PACF** — autocorrelation lags should mostly be inside the confidence band.
3. **Q-Q plot vs normal** — heavy tails would explain why MAPE blows up on a few days.
4. **Distribution** — should be roughly centered.

In [ ]:
import scipy.stats as stats
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

prod_pred = predictions["GradientBoosting (prod)"]
residuals = y_test - prod_pred

fig, axes = plt.subplots(2, 2, figsize=(13, 8))

# 1. Residuals over time
axes[0, 0].plot(y_test.index, residuals, color="#2563eb", linewidth=0.8)
axes[0, 0].axhline(0, color="#64748b", linestyle="--", linewidth=1)
axes[0, 0].set_title("Residuals over time")

# 2. Histogram
axes[0, 1].hist(residuals, bins=40, color="#2563eb", alpha=0.7)
axes[0, 1].set_title("Residual distribution")
axes[0, 1].axvline(0, color="#64748b", linestyle="--")

# 3. Q-Q
stats.probplot(residuals, dist="norm", plot=axes[1, 0])
axes[1, 0].set_title("Q-Q plot vs normal")
axes[1, 0].get_lines()[0].set_markerfacecolor("#2563eb")
axes[1, 0].get_lines()[0].set_markeredgecolor("#2563eb")

# 4. ACF
plot_acf(residuals, lags=30, ax=axes[1, 1])
axes[1, 1].set_title("ACF of residuals (lags 1-30)")

plt.tight_layout(); plt.show()

**What I'd read into these plots:**

- If the residual time series shows a clear trend in the second half, the model is biased on a specific period — usually a sign the training window doesn't match the production distribution.
- Q-Q tails curving away from the line means a normal-distribution assumption (e.g. for confidence intervals) is unsafe. Bootstrap-based intervals would be more honest.
- ACF spikes at lag 7 or 14 would suggest residual seasonality — i.e. the calendar features aren't doing their job and lag features are leaking the same information.

## Drift simulation: train 2010, score 2011

Real production models degrade. This is a quick proxy: train the model on the first calendar year only, then score every month of the second year separately. If MAE rises monotonically, the model needs periodic retraining.

In [ ]:
by_year = features.groupby(features.index.year)
years = sorted(by_year.groups.keys())
if len(years) >= 2:
    early = features[features.index.year == years[0]]
    late = features[features.index.year >= years[1]]

    drift_model = GradientBoostingRegressor(**MODEL_PARAMS)
    drift_model.fit(early[FEATURE_NAMES], early["revenue"])

    monthly_mae = (
        late.assign(
            pred=drift_model.predict(late[FEATURE_NAMES]),
            err=lambda d: (d["revenue"] - d["pred"]).abs(),
            month=lambda d: d.index.to_period("M"),
        )
        .groupby("month")["err"].mean()
    )

    fig, ax = plt.subplots()
    monthly_mae.plot(ax=ax, marker="o", color="#dc2626")
    ax.set_title(f"MAE per month — model trained only on {years[0]}")
    ax.set_ylabel("MAE")
    ax.set_xlabel("")
    plt.tight_layout(); plt.show()
    monthly_mae.round(2).to_frame("MAE")
else:
    print("Dataset only covers a single calendar year — drift simulation skipped.")

**Takeaway:** the slope of the line tells me how often I'd want to retrain in production. If MAE doubles within 6 months, monthly retraining is justified. Online Retail II is short, so the drift here is mostly noise + seasonality interaction, not real concept drift — but the methodology stands.

## Takeaways

- Production model stays gradient boosting **if** the comparison table shows it as the clear winner on the chronological split. If LightGBM wins by a small margin, I'd keep GB — fewer moving parts, easier to explain in interviews.
- SARIMA is worth including in the methodology page even if not deployed, as a comparison point. Reviewers like seeing a statistical baseline.
- Residual diagnostics here would inform whether a bootstrapped prediction interval is honest — that's a follow-up.
- Drift simulation is a tool to revisit before any future production deployment.